# Validation-Set Tuning — Classifier + PM gates

Searches XGBoost classifier hyperparameters (random search) and PM-gate combos
on the **validation** split (untouched until now). Test set stays locked.

## Cache flag
Set `FORCE_RECOMPUTE = True` to rebuild everything. With `False` (default) each
expensive artifact is loaded from `outputs/tuning_cache/` if present.

## Stages
1. Build/load panel covering train+val
2. Build/load per-sector shadow + return model store (fit on TRAIN only)
3. Classifier random search on val (fit train → score val by f1_macro)
4. For top-K classifiers: build val-period `selected_panel` (DBTS walk)
5. PM gate sweep on each val panel; pick best combo by Sharpe
6. Compare to baseline `rz=2.0, conf=0.70, mr_exit=0.30`

In [ ]:
# Cell 2 — Imports, cache flag, paths
import os, sys, json, math, hashlib, pickle, warnings, dataclasses
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
%matplotlib inline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'strategy').exists():
    cand = Path(r'C:\algo-trading-project')
    if cand.exists():
        PROJECT_ROOT = cand
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# ============== CACHE FLAG ==============
FORCE_RECOMPUTE = False
# Per-stage overrides (useful for selective rebuild). Each defaults to FORCE_RECOMPUTE.
RECOMPUTE_PANEL          = FORCE_RECOMPUTE
RECOMPUTE_MODEL_STORE    = FORCE_RECOMPUTE
RECOMPUTE_CLF_SEARCH     = FORCE_RECOMPUTE
RECOMPUTE_VAL_DBTS_PANEL = FORCE_RECOMPUTE
RECOMPUTE_PM_SWEEP       = True   # cheap, always rerun
# =========================================

CACHE_DIR = Path('outputs/tuning_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root      : {PROJECT_ROOT}')
print(f'FORCE_RECOMPUTE   : {FORCE_RECOMPUTE}')
print(f'Cache dir         : {CACHE_DIR.resolve()}')

In [ ]:
# Cell 3 — Project imports + load data + define train_idx / val_idx
from config import SECTORS
from strategy.strategy_config import StrategyConfig
from strategy.pipeline import StrategyPipeline
from strategy.splits import chrono_split, walk_forward_folds
from strategy.predictor_selector import PredictorSelector
from strategy.regressors import DynamicShadowPriceModel, DynamicReturnModel
from strategy.bandit_target_selector import BanditTargetSelector
from strategy.classifier import make_labels
from strategy.position_manager import PositionManager, PositionState

cfg = StrategyConfig(force_recompute=False, make_plots=False)
pipeline = StrategyPipeline(cfg)

print('Loading market data...')
md = pipeline.load_data()
split = chrono_split(md.prices.index, cfg)

train_idx = pd.DatetimeIndex(split.train_idx).sort_values()
val_idx   = pd.DatetimeIndex(split.val_idx).sort_values()
test_idx  = pd.DatetimeIndex(split.test_idx).sort_values()

h = int(max(getattr(cfg, 'label_horizon', 1), getattr(cfg, 'return_horizon', 1)))
train_fit_idx = train_idx[:-h] if len(train_idx) > h else train_idx[:0]
val_fit_idx   = val_idx[:-h]   if len(val_idx) > h   else val_idx[:0]
trainval_idx  = train_idx.union(val_idx)

print(f'TRAIN : {train_idx[0].date()} -> {train_idx[-1].date()} | n={len(train_idx)}')
print(f'VAL   : {val_idx[0].date()} -> {val_idx[-1].date()} | n={len(val_idx)}')
print(f'TEST  : {test_idx[0].date()} -> {test_idx[-1].date()} | n={len(test_idx)}  (LOCKED)')
print(f'horizon buffer h={h}')

In [ ]:
# Cell 4 — Helpers (mirror DBTS notebook)
def safe_price(prices, ticker, date):
    try:
        v = float(prices.at[date, ticker])
        return v if math.isfinite(v) and v > 0 else float('nan')
    except Exception:
        return float('nan')

def max_drawdown(equity):
    if equity.empty:
        return float('nan')
    peak = equity.cummax()
    return float((equity / peak - 1.0).min())

def pm_portfolio_metrics(pm_df, periods_per_year=252):
    if pm_df.empty:
        return pd.Series(dtype=float)
    daily_ret = pm_df.groupby('date')['net_pnl'].mean().sort_index()
    if daily_ret.empty:
        return pd.Series(dtype=float)
    equity = (1.0 + daily_ret).cumprod()
    cum = float(equity.iloc[-1] - 1.0)
    n = len(daily_ret)
    ann_ret = float((1.0 + cum) ** (periods_per_year / max(n, 1)) - 1.0)
    ann_vol = float(daily_ret.std(ddof=1) * np.sqrt(periods_per_year)) if n > 1 else np.nan
    sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) and ann_vol != 0 else np.nan
    dn = daily_ret[daily_ret < 0]
    dn_vol = float(dn.std(ddof=1) * np.sqrt(periods_per_year)) if len(dn) > 1 else np.nan
    sortino = ann_ret / dn_vol if dn_vol and np.isfinite(dn_vol) and dn_vol != 0 else np.nan
    entries = pm_df[pm_df.get('is_entry', pd.Series(False, index=pm_df.index)) == True]
    active  = pm_df[pm_df['position'] != 0]
    return pd.Series({
        'trading_days': n,
        'total_entries': int(len(entries)),
        'long_entries':  int((entries['position'] == 1).sum()) if not entries.empty else 0,
        'short_entries': int((entries['position'] == -1).sum()) if not entries.empty else 0,
        'active_days':   int((active['position'] != 0).sum()),
        'cumulative_return':     round(cum, 4),
        'annualized_return':     round(ann_ret, 4),
        'annualized_volatility': round(ann_vol, 4) if np.isfinite(ann_vol) else np.nan,
        'sharpe':                round(sharpe, 4) if np.isfinite(sharpe) else np.nan,
        'sortino':               round(sortino, 4) if np.isfinite(sortino) else np.nan,
        'max_drawdown':          round(max_drawdown(equity), 4),
        'win_rate_days':         round(float((daily_ret > 0).mean()), 4),
        'avg_daily_pnl':         round(float(daily_ret.mean()), 6),
    })

def _params_hash(d):
    s = json.dumps(d, sort_keys=True, default=str)
    return hashlib.md5(s.encode()).hexdigest()[:10]

print('Helpers ready.')

In [ ]:
# Cell 5 — Build/load panel covering train+val
PANEL_CACHE = CACHE_DIR / 'panel_trainval.pkl'

if PANEL_CACHE.exists() and not RECOMPUTE_PANEL:
    print(f'[CACHE HIT] loading panel from {PANEL_CACHE.name}')
    panel = pd.read_pickle(PANEL_CACHE)
else:
    print('[RECOMPUTE] building panel for train+val...')
    _diag_cfg = dataclasses.replace(cfg, min_train_days=120)
    # walk-forward folds across the full train+val window so the panel covers val too
    tv_fit_idx = trainval_idx[:-h] if len(trainval_idx) > h else trainval_idx[:0]
    folds = walk_forward_folds(tv_fit_idx, _diag_cfg)
    print(f'  folds: {len(folds)}')
    _stale = pipeline.cache._path('feature_panel', 'pkl', 'panel')
    if _stale.exists():
        _stale.unlink()
    panel = pipeline.build_panel(md, folds, split)
    panel['date'] = pd.to_datetime(panel['date'])
    panel = panel[panel['date'].isin(tv_fit_idx)].copy()
    panel.to_pickle(PANEL_CACHE)
    print(f'  saved -> {PANEL_CACHE.name}')

print(f'Panel rows: {len(panel):,}  dates: {panel["date"].nunique()}  '
      f'{panel["date"].min().date()} -> {panel["date"].max().date()}')

# Define feature columns (same exclusions as DBTS notebook)
excluded = (
    'date','etf','sector','target','predictors','target_price','shadow_price',
    'next_ret','label','spread_signal','ann_vol','residual_z','price_residual',
    'residual_ewm_mean','residual_ewm_std','residual_roll_mean','residual_roll_std',
    'predicted_return','candidate_price','dbts_score','was_selected_by_dbts',
)
feature_cols = [c for c in panel.columns
                if c not in excluded and pd.api.types.is_numeric_dtype(panel[c])]
print(f'Classifier feature cols: {len(feature_cols)}')

In [ ]:
# Cell 6 — Build/load model_store (shadow + return models) fit on TRAIN only
MODEL_STORE_CACHE = CACHE_DIR / 'model_store.pkl'

if MODEL_STORE_CACHE.exists() and not RECOMPUTE_MODEL_STORE:
    print(f'[CACHE HIT] loading model_store from {MODEL_STORE_CACHE.name}')
    with open(MODEL_STORE_CACHE, 'rb') as f:
        model_store = pickle.load(f)
else:
    print('[RECOMPUTE] fitting per-sector shadow + return models on TRAIN only...')
    model_store = {}
    for etf, cfg_sector in SECTORS.items():
        sector_name = cfg_sector['name']
        members = [cfg_sector['target']] + cfg_sector['predictors']
        model_store[etf] = {}
        for cand in members:
            peers = [m for m in members if m != cand and m in md.prices.columns]
            if cand not in md.prices.columns or not peers:
                continue
            psel = PredictorSelector(cfg)
            choice = psel.select(cand, peers,
                                 md.returns.reindex(train_fit_idx),
                                 md.prices.loc[train_fit_idx])
            preds = list(choice.selected)
            assert cand not in preds
            shadow_m = DynamicShadowPriceModel(cfg)
            shadow_feats, _, base_price, _ = shadow_m.fit(md.prices, cand, preds, train_fit_idx)
            return_m = DynamicReturnModel(cfg)
            return_feats, _, _ = return_m.fit(md.prices, cand, preds, train_idx)
            model_store[etf][cand] = dict(
                predictors=preds, shadow_model=shadow_m, shadow_feats=shadow_feats,
                base_price=base_price, return_model=return_m, return_feats=return_feats,
            )
        print(f'  {sector_name}: {len(model_store[etf])} candidate models')
    with open(MODEL_STORE_CACHE, 'wb') as f:
        pickle.dump(model_store, f)
    print(f'  saved -> {MODEL_STORE_CACHE.name}')

In [ ]:
# Cell 7 — Random search XGB classifier on validation (fit TRAIN, score VAL)
CLF_SEARCH_CACHE = CACHE_DIR / 'clf_search.pkl'

N_TRIALS = 15           # random-search budget
TOP_K    = 3            # top configs taken to PM-stage
SEED     = int(getattr(cfg, 'random_state', 42))

if CLF_SEARCH_CACHE.exists() and not RECOMPUTE_CLF_SEARCH:
    print(f'[CACHE HIT] loading clf search from {CLF_SEARCH_CACHE.name}')
    with open(CLF_SEARCH_CACHE, 'rb') as f:
        clf_search = pickle.load(f)
    clf_results_df = clf_search['results']
    top_models     = clf_search['top_models']
else:
    print(f'[RECOMPUTE] random search: {N_TRIALS} trials, picking top {TOP_K}')
    data = panel.dropna(subset=['label']).copy()
    X_all = data[feature_cols].apply(pd.to_numeric, errors='coerce')
    X_all = X_all.groupby(data['target']).ffill().fillna(0.0)
    for c in feature_cols:
        data[c] = X_all[c]
    train_mask = data['date'].isin(train_fit_idx)
    val_mask   = data['date'].isin(val_fit_idx)
    X_tr, y_tr = data.loc[train_mask, feature_cols], data.loc[train_mask, 'label']
    X_va, y_va = data.loc[val_mask,   feature_cols], data.loc[val_mask,   'label']
    print(f'  train rows: {len(X_tr):,}  val rows: {len(X_va):,}')
    if len(X_va) == 0:
        raise RuntimeError('No val rows — check val_fit_idx vs panel coverage')

    _TO_IDX = {-1: 0, 0: 1, 1: 2}
    yi_tr = y_tr.map(_TO_IDX).astype(int)
    yi_va = y_va.map(_TO_IDX).astype(int)
    sw_tr = compute_sample_weight(class_weight='balanced', y=yi_tr)

    rng = np.random.RandomState(SEED)
    base = dict(cfg.clf_params)
    base.update({'num_class': 3, 'objective': 'multi:softprob', 'verbosity': 0,
                 'random_state': SEED, 'n_jobs': 1})
    trials = []
    for t in range(N_TRIALS):
        params = {**base,
                  'n_estimators':     int(rng.choice([200, 300, 400])),
                  'learning_rate':    float(rng.choice([0.02, 0.03, 0.05])),
                  'max_depth':        int(rng.choice([2, 3, 4, 5])),
                  'subsample':        float(rng.choice([0.7, 0.8, 0.9, 1.0])),
                  'colsample_bytree': float(rng.choice([0.7, 0.8, 0.9, 1.0])),
                  'reg_lambda':       float(rng.choice([0.5, 1.0, 2.0, 5.0])),
                  'reg_alpha':        float(rng.choice([0.0, 0.1, 0.5, 1.0]))}
        m = XGBClassifier(**params)
        m.fit(X_tr, yi_tr, sample_weight=sw_tr)
        pred_va = m.predict(X_va)
        f1   = f1_score(yi_va, pred_va, average='macro')
        acc  = accuracy_score(yi_va, pred_va)
        trials.append({
            'trial': t, 'f1_macro_val': round(f1, 4), 'accuracy_val': round(acc, 4),
            **{k: params[k] for k in ['n_estimators','learning_rate','max_depth',
                                       'subsample','colsample_bytree','reg_lambda','reg_alpha']},
            'model': m, 'params': params,
        })
        print(f'  trial {t+1:2}/{N_TRIALS}  f1_macro={f1:.4f}  acc={acc:.4f}  '
              f'depth={params["max_depth"]} lr={params["learning_rate"]} '
              f'n={params["n_estimators"]}')
    clf_results_df = (pd.DataFrame([{k: v for k, v in t.items() if k not in ("model","params")}
                                    for t in trials])
                       .sort_values('f1_macro_val', ascending=False)
                       .reset_index(drop=True))
    top_idx = clf_results_df['trial'].head(TOP_K).tolist()
    top_models = [{'trial': t, 'model': trials[t]['model'], 'params': trials[t]['params'],
                   'features': feature_cols,
                   'f1_macro_val': trials[t]['f1_macro_val']} for t in top_idx]
    with open(CLF_SEARCH_CACHE, 'wb') as f:
        pickle.dump({'results': clf_results_df, 'top_models': top_models}, f)
    print(f'  saved -> {CLF_SEARCH_CACHE.name}')

print('\nClassifier search results (sorted by val f1_macro):')
display(clf_results_df)
print(f'\nTop {len(top_models)} taken to PM stage: trials {[m["trial"] for m in top_models]}')

In [ ]:
# Cell 8 — Build val-period selected_panel for each top classifier (DBTS walk)
# Output stored per classifier trial in CACHE_DIR / f'val_dbts_t{trial}.pkl'
from statsmodels.tsa.stattools import adfuller

DBTS_WEIGHTS = {'bandit': 0.40, 'residual': 0.25, 'pred_ret': 0.20, 'adf': 0.15}
ADF_RECOMPUTE_EVERY = 20

panel_indexed = panel.set_index(['date', 'target'])
_has_pred_ret = 'predicted_return' in panel_indexed.columns
_panel_dates  = set(panel_indexed.index.get_level_values('date'))
eval_dates    = [d for d in val_fit_idx if d in _panel_dates]
train_idx_set = set(train_idx)
print(f'val eval dates: {len(eval_dates)} '
      f'({pd.Timestamp(eval_dates[0]).date()} -> {pd.Timestamp(eval_dates[-1]).date()})')

def _normalize_signal(raw):
    try:
        s = int(raw)
    except Exception:
        return 0
    if s in (-1, 0, 1): return s
    if s in (0, 1, 2):  return {0:-1, 1:0, 2:1}[s]
    return 0

def _build_val_selected_panel(model, features, trial_id):
    cache_path = CACHE_DIR / f'val_dbts_t{trial_id}.pkl'
    if cache_path.exists() and not RECOMPUTE_VAL_DBTS_PANEL:
        print(f'  [CACHE HIT] {cache_path.name}')
        return pd.read_pickle(cache_path)
    print(f'  [RECOMPUTE] trial {trial_id} val DBTS walk...')
    bandit = BanditTargetSelector(cfg)
    for etf, cfg_sector in SECTORS.items():
        sector_name = cfg_sector['name']
        members = [cfg_sector['target']] + cfg_sector['predictors']
        bandit.init_sector(sector_name, members)

    adf_cache = {}
    last_selected = {}
    selected_rows = []

    for date_no, date in enumerate(eval_dates, start=1):
        for etf, cfg_sector in SECTORS.items():
            sector_name = cfg_sector['name']
            members = [cfg_sector['target']] + cfg_sector['predictors']
            bsamp = bandit.sample_scores(sector_name)
            scores, comp = {}, {}
            for cand in members:
                rec = model_store.get(etf, {}).get(cand)
                if rec is None or cand not in md.prices.columns:
                    scores[cand] = -np.inf; continue
                rz, pr = np.nan, np.nan
                resid = pd.Series(dtype=float)
                try:
                    pk = (date, cand)
                    if pk in panel_indexed.index:
                        r = panel_indexed.loc[pk]
                        if 'residual_z' in panel_indexed.columns:
                            v = r['residual_z'] if isinstance(r, pd.Series) else r.get('residual_z', np.nan)
                            rz = float(v) if pd.notna(v) else np.nan
                        if _has_pred_ret:
                            v = r['predicted_return'] if isinstance(r, pd.Series) else r.get('predicted_return', np.nan)
                            pr = float(v) if pd.notna(v) else np.nan
                    ck = (sector_name, cand)
                    if ck not in adf_cache or date_no % ADF_RECOMPUTE_EVERY == 0:
                        feats = rec['shadow_feats']
                        pidx  = pd.DatetimeIndex([d for d in feats.loc[:date].index
                                                  if d in train_idx_set and d <= date])
                        if len(pidx) > 0:
                            sp = rec['shadow_model'].predict(feats, pidx, rec['base_price'])
                            ph = md.prices[cand].reindex(pidx)
                            resid = (ph - sp).dropna()
                except Exception:
                    pass
                rs = min(abs(rz)/3.0, 1.0) if np.isfinite(rz) else 0.0
                ps = abs(max(-1.0, min(1.0, pr/0.05))) if np.isfinite(pr) else 0.0
                ck = (sector_name, cand)
                if ck not in adf_cache or date_no % ADF_RECOMPUTE_EVERY == 0:
                    try:
                        ser = resid.dropna() if len(resid.dropna()) >= 30 else (
                            md.prices[cand].loc[:date].dropna().astype(float)
                            .pipe(lambda s: s[s > 0].tail(120))
                            .pipe(lambda s: s - s.rolling(30, min_periods=10).mean())
                            .dropna())
                        ap = float(adfuller(ser, autolag='AIC')[1]) if len(ser) >= 30 else np.nan
                    except Exception:
                        ap = np.nan
                    adf_cache[ck] = ap
                else:
                    ap = adf_cache[ck]
                asc = 1.0 - min(ap, 1.0) if np.isfinite(ap) else 0.0
                bs  = float(bsamp.get(cand, 0.5))
                fs  = (DBTS_WEIGHTS['bandit']*bs + DBTS_WEIGHTS['residual']*rs
                       + DBTS_WEIGHTS['pred_ret']*ps + DBTS_WEIGHTS['adf']*asc)
                scores[cand] = fs
                comp[cand] = dict(residual_z=rz, bandit_score=bs, adf_p=ap)
            finite = {k: v for k, v in scores.items() if np.isfinite(v)}
            selected = max(finite, key=finite.get) if finite else members[0]
            last_selected[sector_name] = selected

            # classifier predict from panel row
            pk = (date, selected)
            if pk in panel_indexed.index:
                pr_row = panel_indexed.loc[pk]
                fd = {c: 0.0 for c in features}
                for c in features:
                    v = pr_row[c] if c in pr_row.index else 0.0
                    try:
                        fv = float(v); fd[c] = fv if np.isfinite(fv) else 0.0
                    except (TypeError, ValueError):
                        fd[c] = 0.0
                Xrow = pd.DataFrame([fd])
                proba = model.predict_proba(Xrow)
                pred  = _normalize_signal(int(model.predict(Xrow)[0]))
                p_short, p_flat, p_long = [float(x) for x in proba[0].tolist()]
                rz_p = float(pr_row.get('residual_z', np.nan)) if hasattr(pr_row, 'get') else (
                       float(pr_row['residual_z']) if 'residual_z' in pr_row.index else np.nan)
                nr   = float(pr_row.get('next_ret', np.nan)) if hasattr(pr_row, 'get') else (
                       float(pr_row['next_ret']) if 'next_ret' in pr_row.index else np.nan)
            else:
                pred, p_short, p_flat, p_long = 0, 0.333, 0.334, 0.333
                rz_p, nr = np.nan, np.nan
            selected_rows.append({
                'date': date, 'sector': sector_name, 'etf': etf, 'target': selected,
                'target_price': safe_price(md.prices, selected, date),
                'signal': pred, 'P_short': p_short, 'P_flat': p_flat, 'P_long': p_long,
                'residual_z': rz_p, 'next_ret': nr,
                'final_target_score': scores.get(selected, np.nan),
            })
    sp = pd.DataFrame(selected_rows)
    sp.to_pickle(cache_path)
    print(f'  saved -> {cache_path.name}  rows={len(sp):,}')
    return sp

val_panels = {}
for tm in top_models:
    print(f'\n=== trial {tm["trial"]} (f1_val={tm["f1_macro_val"]}) ===')
    val_panels[tm['trial']] = _build_val_selected_panel(tm['model'], tm['features'], tm['trial'])
print('\nAll val selected_panels ready.')

In [ ]:
# Cell 9 — PM gate sweep on each val selected_panel
PM_COST = 5.0 / 1e4
ENTRY_ACTIONS = {'ENTER_LONG','ENTER_SHORT','FLIP_LONG_TO_SHORT','FLIP_SHORT_TO_LONG'}
EXIT_ACTIONS  = {'EXIT','FLIP_LONG_TO_SHORT','FLIP_SHORT_TO_LONG'}
MIN_TARGET_HOLD_DAYS = 5
TARGET_SWITCH_MARGIN = 0.05

PM_GRID = {
    'rz_thr':  [1.5, 2.0, 2.5],
    'conf':    [0.65, 0.70, 0.75],
    'mr_exit': [0.20, 0.30],
}
FIXED = dict(flat_block=0.25, max_hold=10, allow_flip=False)

def simulate_pm(sp, rz_thr, conf, mr_exit):
    pm = PositionManager(
        long_entry_confidence=conf, short_entry_confidence=conf,
        flat_probability_block=FIXED['flat_block'],
        entry_residual_threshold=rz_thr,
        mean_reversion_exit=mr_exit,
        opposite_signal_confidence=0.70,
        stop_loss=-0.02, take_profit=0.03,
        max_holding_days=FIXED['max_hold'], allow_flip=FIXED['allow_flip'],
    )
    sp_sorted = sp.sort_values(['date','sector']).reset_index(drop=True)
    state_by_sec, prev_pos_by_sec, trade_seq = {}, {}, {}
    cur_tgt, hold_days = {}, {}
    rows = []
    for date, day_df in sp_sorted.groupby('date', sort=True):
        for _, r in day_df.iterrows():
            sec = r['sector']; selected_top = r['target']
            cur = cur_tgt.get(sec); held = hold_days.get(sec, 0)
            if cur is None or cur == selected_top:
                selected = selected_top
            elif held < MIN_TARGET_HOLD_DAYS:
                selected = cur
            elif r.get('final_target_score', -np.inf) < -np.inf + TARGET_SWITCH_MARGIN:
                selected = cur
            else:
                selected = selected_top
            switched = (cur is not None) and (selected != cur)
            hold_days[sec] = 1 if switched else (held + 1)
            prev_pos = prev_pos_by_sec.get(sec, 0)
            if switched and prev_pos != 0:
                # forced exit (approx: no next_ret available for prev target → cost only)
                turn = abs(0 - prev_pos)
                net  = -turn * PM_COST
                st = state_by_sec.get(sec, PositionState())
                rows.append({'date': date, 'sector': sec, 'target': cur,
                             'action': 'EXIT', 'position': 0.0, 'prev_pos': float(prev_pos),
                             'gross_pnl': 0.0, 'net_pnl': net,
                             'is_entry': False, 'is_exit': True})
                state_by_sec[sec] = PositionState()
                prev_pos_by_sec[sec] = 0; prev_pos = 0
            cur_tgt[sec] = selected
            state = state_by_sec.get(sec) or PositionState()
            nseq  = trade_seq.get(sec, 0)
            nr    = float(r['next_ret']) if pd.notna(r['next_ret']) else 0.0
            row_in = pd.Series({
                'date': date, 'target': selected, 'sector': sec,
                'signal': int(r['signal']),
                'P_short': float(r['P_short']), 'P_flat': float(r['P_flat']),
                'P_long':  float(r['P_long']),
                'residual_z': float(r['residual_z']) if pd.notna(r['residual_z']) else 0.0,
                'next_ret': nr, 'target_price': r['target_price'],
            })
            d = pm.decide(row_in, state)
            pos = d.position; turn = abs(pos - prev_pos)
            gross = pos * nr; net = gross - turn * PM_COST
            is_e = d.action in ENTRY_ACTIONS; is_x = d.action in EXIT_ACTIONS
            rows.append({'date': date, 'sector': sec, 'target': selected,
                         'action': d.action, 'position': float(pos),
                         'prev_pos': float(prev_pos),
                         'gross_pnl': gross, 'net_pnl': net,
                         'is_entry': bool(is_e), 'is_exit': bool(is_x)})
            if is_e:
                nseq += 1
                state_by_sec[sec] = PositionState(
                    current_position=int(pos), days_in_position=1,
                    entry_residual_z=row_in['residual_z'],
                    entry_confidence=r['P_long'] if pos == 1 else r['P_short'],
                    trade_pnl=net, trade_id=nseq)
            elif pos == 0:
                state_by_sec[sec] = PositionState()
            elif pos == prev_pos and prev_pos != 0:
                state.current_position = pos
                state.days_in_position += 1
                state.trade_pnl += net
                state_by_sec[sec] = state
            prev_pos_by_sec[sec] = pos; trade_seq[sec] = nseq
    return pd.DataFrame(rows)

sweep_rows = []
for tm in top_models:
    sp = val_panels[tm['trial']]
    for rz_thr, conf, mre in product(PM_GRID['rz_thr'], PM_GRID['conf'], PM_GRID['mr_exit']):
        pm_df = simulate_pm(sp, rz_thr, conf, mre)
        m = pm_portfolio_metrics(pm_df) if not pm_df.empty else pd.Series(dtype=float)
        sweep_rows.append({
            'clf_trial': tm['trial'], 'f1_val': tm['f1_macro_val'],
            'rz_thr': rz_thr, 'conf': conf, 'mr_exit': mre,
            **{k: m.get(k) for k in ['total_entries','sharpe','sortino','cumulative_return',
                                      'annualized_return','annualized_volatility',
                                      'max_drawdown','win_rate_days','active_days']},
        })
        print(f"  t{tm['trial']}  rz={rz_thr} conf={conf} mre={mre} | "
              f"entries={int(m.get('total_entries',0))} "
              f"sharpe={m.get('sharpe',np.nan):+.2f} "
              f"cumret={m.get('cumulative_return',np.nan):+.3f}")

sweep_df = pd.DataFrame(sweep_rows).sort_values('sharpe', ascending=False).reset_index(drop=True)
sweep_df.to_csv(CACHE_DIR / 'pm_sweep_val.csv', index=False)
print(f'\nSaved sweep -> {CACHE_DIR/"pm_sweep_val.csv"}')

# --- Append to persistent run history ---
from datetime import datetime
RUN_HISTORY = CACHE_DIR / 'run_history.csv'
_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
_hist_cols = ['timestamp','source','tag','clf_trial','f1_val','rz_thr','conf','mr_exit',
              'total_entries','sharpe','sortino','cumulative_return',
              'annualized_return','annualized_volatility','max_drawdown',
              'win_rate_days','active_days']
_hist_rows = []
for _, r in sweep_df.iterrows():
    _hist_rows.append({
        'timestamp': _ts,
        'source': 'Validation_Tuning',
        'tag': f"t{int(r['clf_trial'])}_rz{r['rz_thr']}_c{r['conf']}_mre{r['mr_exit']}",
        **{k: r.get(k) for k in _hist_cols if k not in ('timestamp','source','tag')},
    })
_new_hist = pd.DataFrame(_hist_rows)[_hist_cols]
if RUN_HISTORY.exists():
    _prev = pd.read_csv(RUN_HISTORY)
    _combined = pd.concat([_prev, _new_hist], ignore_index=True)
else:
    _combined = _new_hist
_combined.to_csv(RUN_HISTORY, index=False)
print(f'Appended {len(_new_hist)} rows to {RUN_HISTORY.name} (total: {len(_combined)})')

print('\nTop 10 combos by Sharpe (validation):')
display(sweep_df.head(10))

In [ ]:
# Cell 10 — Best combo + baseline comparison
best = sweep_df.iloc[0]
baseline = sweep_df.query('rz_thr == 2.0 and conf == 0.70 and mr_exit == 0.30')
if not baseline.empty:
    baseline = baseline.iloc[baseline['sharpe'].argmax()] if len(baseline) > 1 else baseline.iloc[0]

print('=' * 70)
print('BEST COMBO ON VALIDATION')
print('=' * 70)
print(f"clf trial   : {int(best['clf_trial'])}  (val f1_macro={best['f1_val']})")
print(f"PM params   : rz_thr={best['rz_thr']}  conf={best['conf']}  mr_exit={best['mr_exit']}")
print(f"entries     : {int(best['total_entries'])}")
print(f"Sharpe      : {best['sharpe']:+.3f}")
print(f"Sortino     : {best['sortino']:+.3f}")
print(f"cum return  : {best['cumulative_return']:+.3f}")
print(f"ann return  : {best['annualized_return']:+.3f}")
print(f"ann vol     : {best['annualized_volatility']:+.3f}")
print(f"max DD      : {best['max_drawdown']:+.3f}")
print(f"win rate    : {best['win_rate_days']:+.3f}")

if isinstance(baseline, pd.Series):
    print('\n' + '=' * 70)
    print('BASELINE (rz=2.0, conf=0.70, mr_exit=0.30) for reference')
    print('=' * 70)
    print(f"clf trial   : {int(baseline['clf_trial'])}")
    print(f"entries     : {int(baseline['total_entries'])}")
    print(f"Sharpe      : {baseline['sharpe']:+.3f}")
    print(f"Sortino     : {baseline['sortino']:+.3f}")
    print(f"cum return  : {baseline['cumulative_return']:+.3f}")
    print(f"max DD      : {baseline['max_drawdown']:+.3f}")

# Best classifier params
best_clf_params = next(tm['params'] for tm in top_models if tm['trial'] == int(best['clf_trial']))
tuned_only = {k: best_clf_params[k] for k in
              ['n_estimators','learning_rate','max_depth','subsample',
               'colsample_bytree','reg_lambda','reg_alpha']}
print('\nBest classifier hyperparameters:')
for k, v in tuned_only.items():
    print(f'  {k:20s} = {v}')

In [ ]:
# Cell 11 — Show full run history (all sweeps + external runs from _log_run.py)
RUN_HISTORY = CACHE_DIR / 'run_history.csv'
if RUN_HISTORY.exists():
    hist = pd.read_csv(RUN_HISTORY)
    print(f'Total runs logged: {len(hist)}')
    print(f'Sources: {hist["source"].value_counts().to_dict()}')
    print('\nTop 15 by Sharpe across ALL runs:')
    display(hist.sort_values("sharpe", ascending=False).head(15))
    print('\nMost recent 10 runs:')
    display(hist.tail(10))
else:
    print('No history yet — run sweep cell first or use _log_run.py')
